In [1]:
!pip -q install -U cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
import cohere
from getpass import getpass

api_key = getpass("Enter your Cohere API key: ")
co = cohere.ClientV2(api_key=api_key)

MODEL = "command-a-03-2025"

print("Cohere client initialized.")

Enter your Cohere API key: ··········
Cohere client initialized.


In [3]:

# LAB 1: MULTI-AGENT ORCHESTRATION

def ask_cohere(role, task):
    response = co.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": role
            },
            {
                "role": "user",
                "content": task
            }
        ],
        temperature=0.2
    )
    return response.message.content[0].text


# Agent 1: Research Agent
def research_agent(topic):
    return ask_cohere(
        "You are an AI research assistant. "
        "Identify key concepts, applications, limitations, "
        "and research questions. Do not invent citations.",
        f"Research topic: {topic}"
    )


# Agent 2: Teaching Agent
def teaching_agent(topic, research_output):
    return ask_cohere(
        "You are a university professor who designs "
        "technical lectures for AI and Data Science students.",
        f"""
        Topic: {topic}

        Research agent output:
        {research_output}

        Create:
        1. Four learning objectives
        2. A lecture outline
        3. Two classroom activities
        4. One assessment question
        """
    )


# Agent 3: Review Agent
def review_agent(topic, lecture_plan):
    return ask_cohere(
        "You are a critical academic reviewer. "
        "Identify unsupported claims, missing concepts, "
        "ethical concerns, and technical inaccuracies. "
        "Do not assume the draft is correct.",
        f"Topic: {topic}\nLecture plan:\n{lecture_plan}"
    )


# Coordinator: controls the workflow
def run_multi_agent_system(topic):
    print("Running Research Agent...")
    research = research_agent(topic)

    print("\nRunning Teaching Agent...")
    teaching = teaching_agent(topic, research)

    print("\nRunning Review Agent...")
    review = review_agent(topic, teaching)

    print("\n" + "=" * 60)
    print("RESEARCH OUTPUT")
    print("=" * 60)
    print(research)

    print("\nLECTURE PLAN")
    print("=" * 60)
    print(teaching)

    print("\nREVIEW REPORT")
    print("=" * 60)
    print(review)

    return {
        "research": research,
        "teaching": teaching,
        "review": review
    }


# Run the system
results = run_multi_agent_system(
    "Explainable AI in healthcare"
)


Running Research Agent...

Running Teaching Agent...

Running Review Agent...

RESEARCH OUTPUT
**Key Concepts:**
- **Explainable AI (XAI):** A set of techniques and methods aimed at making the decisions and predictions of artificial intelligence systems understandable to human users. In the context of healthcare, XAI ensures that AI models provide transparent and interpretable insights, which is crucial for building trust and facilitating clinical adoption.
- **Interpretability vs. Explainability:** These terms are often used interchangeably, but they have distinct meanings. Interpretability refers to the inherent ability of a model to be understood, while explainability involves post-hoc methods to describe a model's behavior. In healthcare, both are essential for different scenarios, such as understanding a diagnosis or justifying treatment recommendations.
- **Transparency and Trust:** XAI aims to bridge the gap between complex AI models and healthcare professionals by providing cle

In [4]:

import gradio as gr

def research_agent(topic):
    return ask_cohere(
        "You are an AI research assistant. Identify key concepts, "
        "applications, limitations, and research questions. "
        "Do not invent citations.",
        f"Research this topic: {topic}"
    )

def teaching_agent(topic, research):
    return ask_cohere(
        "You are a university professor designing an AI lecture.",
        f"""
        Topic: {topic}
        Research findings: {research}

        Create four learning objectives, a lecture outline,
        two classroom activities, and one assessment question.
        """
    )

def review_agent(topic, lecture):
    return ask_cohere(
        "You are a critical academic reviewer. Identify missing "
        "concepts, unsupported claims, technical inaccuracies, "
        "and ethical concerns.",
        f"Topic: {topic}\nLecture draft:\n{lecture}"
    )

def run_agents(topic):
    if not topic.strip():
        return "Enter a topic.", "", ""

    research = research_agent(topic)
    lecture = teaching_agent(topic, research)
    review = review_agent(topic, lecture)

    return research, lecture, review

with gr.Blocks() as demo:
    gr.Markdown("# Multi-Agent Academic Research Assistant")
    topic = gr.Textbox(
        label="Research or teaching topic",
        value="Explainable AI in healthcare"
    )
    run = gr.Button("Run Multi-Agent Workflow", variant="primary")

    research_out = gr.Markdown(label="Research Agent")
    teaching_out = gr.Markdown(label="Teaching Agent")
    review_out = gr.Markdown(label="Review Agent")

    run.click(
        fn=run_agents,
        inputs=topic,
        outputs=[research_out, teaching_out, review_out]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7b688952e5d2506da8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
